# M7-B1 — Mesures d audit (à compléter)

## 1. Disparate impact du modèle, puis investigation

DI sur les étiquettes et sur les prédictions, puis investigation : étiquette confrontée à la durée réelle, erreurs et calibration par groupe contre les étiquettes `sejour_prolonge`.

In [1]:
import statistics
import subprocess
import sys
import tempfile
import time
from pathlib import Path

import joblib
import pandas as pd
import psutil
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

DATA_PATH = Path("../data/dms_dataset.csv")
MODEL_PATH = Path("../legacy/dms_predictor_v1.joblib")
REPO_ROOT = Path("..").resolve()
FEATURES = ["age", "nb_comorbidites", "imc", "sexe_bin"]

df = pd.read_csv(DATA_PATH)
model = joblib.load(MODEL_PATH)

df["sexe_bin"] = (df["sexe"] == "M").astype(int)
df["proba"] = model.predict_proba(df[FEATURES])[:, 1]
df["pred"] = (df["proba"] >= 0.5).astype(int)

print(df.shape)

(10000, 13)


### 1.1 Disparate impact : étiquettes puis prédictions

DI = taux de signalement « séjour prolongé » du groupe F divisé par celui du groupe M. Repère d'alerte conventionnel : 0,80. C'est un signal, pas un verdict.

In [2]:
def disparate_impact(rates: pd.Series, group_a: str = "F", group_b: str = "M") -> float:
    return rates[group_a] / rates[group_b]


rates = df.groupby("sexe")[["sejour_prolonge", "pred"]].mean()
print(rates.round(3))
print("DI F/M sur les étiquettes :", round(disparate_impact(rates["sejour_prolonge"]), 3))
print("DI F/M sur les prédictions :", round(disparate_impact(rates["pred"]), 3))

      sejour_prolonge   pred
sexe                        
F               0.321  0.141
M               0.491  0.486
DI F/M sur les étiquettes : 0.653
DI F/M sur les prédictions : 0.291


### 1.2 Investigation : la durée réelle explique-t-elle l'écart ?

Si les femmes étaient signalées moins souvent parce que leurs séjours sont plus courts, l'écart serait compréhensible.

In [3]:
print(df.groupby("sexe")["dms_jours"].describe().round(2))

       count  mean   std  min  25%  50%  75%   max
sexe                                              
F     5011.0  5.62  2.46  1.0  3.9  5.6  7.3  14.9
M     4989.0  5.59  2.45  1.0  3.8  5.5  7.3  15.3


### 1.3 Investigation : à durée réelle égale, l'étiquette est-elle attribuée de la même façon ?

Une durée longue ne fait pas à elle seule un séjour prolongé. On compare donc seulement la part d'étiquettes « prolongé » par tranche de durée réelle et par sexe, sans considérer la durée comme une vérité de référence.

In [4]:
DUREE_BINS = [0, 5.5, 7, 9, 16]

df["tranche_duree"] = pd.cut(df["dms_jours"], DUREE_BINS)

label_by_duration = df.pivot_table(
    index="tranche_duree",
    columns="sexe",
    values="sejour_prolonge",
    aggfunc="mean",
    observed=True,
)
print(label_by_duration.round(3))

print(df.groupby(["sexe", "sejour_prolonge"])["dms_jours"].agg(["min", "max"]).round(1))

sexe               F    M
tranche_duree            
(0.0, 5.5]     0.000  0.0
(5.5, 7.0]     0.634  1.0
(7.0, 9.0]     0.629  1.0
(9.0, 16.0]    0.658  1.0
                      min   max
sexe sejour_prolonge           
F    0                1.0  14.9
     1                5.6  14.3
M    0                1.0   5.5
     1                5.6  15.3


### 1.4 Erreurs par groupe : FNR et FPR du modèle contre les étiquettes

FNR : cas étiquetés « prolongé » que le modèle ne signale pas. FPR : cas étiquetés « standard » que le modèle signale. Ces taux mesurent la fidélité du modèle à ses étiquettes, pas leur justesse.

In [5]:
def error_rates(group: pd.DataFrame, reference: str = "sejour_prolonge", prediction: str = "pred") -> pd.Series:
    positives = group[group[reference] == 1]
    negatives = group[group[reference] == 0]
    return pd.Series(
        {
            "FNR": 1 - positives[prediction].mean(),
            "FPR": negatives[prediction].mean(),
        }
    )


errors = pd.DataFrame({sexe: error_rates(group) for sexe, group in df.groupby("sexe")}).T
print(errors.round(3))

     FNR    FPR
F  0.628  0.033
M  0.242  0.223


### 1.5 Calibration par groupe

Probabilité moyenne prédite comparée au taux d'étiquettes « prolongé », pour chaque sexe.

In [6]:
calibration = pd.DataFrame(
    {
        "proba_moyenne": df.groupby("sexe")["proba"].mean(),
        "taux_etiquette": df.groupby("sexe")["sejour_prolonge"].mean(),
    }
)
calibration["ecart"] = calibration["proba_moyenne"] - calibration["taux_etiquette"]
print(calibration.round(3))

      proba_moyenne  taux_etiquette  ecart
sexe                                      
F             0.321           0.321  0.000
M             0.491           0.491 -0.001


## 2. Ressources (psutil)

### 2.1 Coût d'un appel réel à `predict.py`

Temps et pic de mémoire (RSS) du script de production, lancé comme en production depuis la racine du dépôt. Chaque appel recharge Python, scikit-learn et le modèle.

In [7]:
PREDICT_COMMAND = [sys.executable, "legacy/predict.py", "70", "3", "28.5", "1"]


def run_and_measure(command: list[str], cwd: Path, interval: float = 0.005) -> tuple[float, float]:
    start = time.perf_counter()
    process = psutil.Popen(command, cwd=cwd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    peak = 0
    while process.poll() is None:
        try:
            peak = max(peak, process.memory_info().rss)
        except psutil.Error:
            break
        time.sleep(interval)
    return time.perf_counter() - start, peak / 1e6


calls = [run_and_measure(PREDICT_COMMAND, REPO_ROOT) for _ in range(5)]
print(f"temps par appel (médiane, 5 appels) : {statistics.median(c[0] for c in calls):.2f} s")
print(f"pic de mémoire RSS par appel (médiane) : {statistics.median(c[1] for c in calls):.0f} Mo")

temps par appel (médiane, 5 appels) : 0.85 s
pic de mémoire RSS par appel (médiane) : 170 Mo


### 2.2 Taille du modèle et temps d'inférence par volume

Le fichier `.joblib` déployé et le temps de `predict_proba` sur 100, 1 000 et 10 000 lignes (médiane de 5 mesures).

In [8]:
VOLUMES = (100, 1_000, 10_000)


def inference_times(fitted_model, features: pd.DataFrame, runs: int = 5) -> dict[int, float]:
    result = {}
    for volume in VOLUMES:
        sample = features.sample(volume, replace=True, random_state=0)
        durations = []
        for _ in range(runs):
            start = time.perf_counter()
            fitted_model.predict_proba(sample)
            durations.append((time.perf_counter() - start) * 1000)
        result[volume] = statistics.median(durations)
    return result


print(f"taille du modèle déployé : {MODEL_PATH.stat().st_size / 1e6:.1f} Mo")
for volume, duration in inference_times(model, df[FEATURES]).items():
    print(f"inférence sur {volume:>6} lignes : {duration:.1f} ms")

taille du modèle déployé : 5.0 Mo
inférence sur    100 lignes : 1.0 ms
inférence sur   1000 lignes : 3.8 ms
inférence sur  10000 lignes : 28.4 ms


### 2.3 Temps d'entraînement

Réentraînement avec les paramètres exacts de `train.py` sur les 10 000 séjours (médiane de 3 entraînements).

In [9]:
def train_legacy() -> RandomForestClassifier:
    return RandomForestClassifier(n_estimators=60, max_depth=10, random_state=0).fit(df[FEATURES], df["sejour_prolonge"])


train_durations = []
for _ in range(3):
    start = time.perf_counter()
    train_legacy()
    train_durations.append(time.perf_counter() - start)

print(f"temps d'entraînement (médiane, 3 essais) : {statistics.median(train_durations):.2f} s")

temps d'entraînement (médiane, 3 essais) : 0.18 s


## 3. Comparaison à 2 alternatives

### 3.1 Comparaison sur les mêmes données

Le modèle hérité (mêmes paramètres, réentraîné) est comparé à une régression logistique et à un `HistGradientBoostingClassifier`, sur les mêmes variables et le même découpage 80/20. F1 et accuracy sont mesurés sur les 20 % non vus. Le modèle déployé, lui, n'a jamais été évalué ainsi : on donne aussi son score sur ses données d'entraînement.

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    df[FEATURES], df["sejour_prolonge"], test_size=0.2, random_state=0, stratify=df["sejour_prolonge"]
)

CANDIDATES = {
    "RandomForest (paramètres hérités)": RandomForestClassifier(n_estimators=60, max_depth=10, random_state=0),
    "Régression logistique": LogisticRegression(max_iter=1000),
    "HistGradientBoosting": HistGradientBoostingClassifier(random_state=0),
}

SINGLE_CALL_CODE = (
    "import joblib, pandas as pd;"
    "m = joblib.load({path!r});"
    "m.predict_proba(pd.DataFrame([[70, 3, 28.5, 1]], columns={features!r}))"
)


def evaluate(name: str, candidate, directory: Path) -> dict:
    start = time.perf_counter()
    candidate.fit(X_train, y_train)
    train_seconds = time.perf_counter() - start

    predictions = candidate.predict(X_test)
    model_path = directory / f"{name}.joblib"
    joblib.dump(candidate, model_path)

    call_command = [sys.executable, "-c", SINGLE_CALL_CODE.format(path=str(model_path), features=FEATURES)]
    call_measures = [run_and_measure(call_command, REPO_ROOT) for _ in range(3)]

    return {
        "modele": name,
        "accuracy_test": accuracy_score(y_test, predictions),
        "f1_test": f1_score(y_test, predictions),
        "entrainement_s": train_seconds,
        "inference_10k_ms": inference_times(candidate, df[FEATURES])[10_000],
        "taille_mo": model_path.stat().st_size / 1e6,
        "appel_s": statistics.median(c[0] for c in call_measures),
        "appel_rss_mo": statistics.median(c[1] for c in call_measures),
    }


with tempfile.TemporaryDirectory() as tmp:
    comparison = pd.DataFrame([evaluate(name, candidate, Path(tmp)) for name, candidate in CANDIDATES.items()])

print(comparison.set_index("modele").round(3).to_string())

                                   accuracy_test  f1_test  entrainement_s  inference_10k_ms  taille_mo  appel_s  appel_rss_mo
modele                                                                                                                       
RandomForest (paramètres hérités)          0.680    0.567           0.143            28.459      4.632    0.885       168.608
Régression logistique                      0.686    0.576           0.007             0.226      0.001    0.785       151.241
HistGradientBoosting                       0.684    0.581           0.356             7.136      0.366    0.899       161.235


### 3.2 Le modèle déployé, sur ses propres données d'entraînement

Pour situer l'écart entre la performance affichée par `train.py` et la performance sur des données non vues.

In [11]:
deployed_predictions = model.predict(df[FEATURES])
print(f"accuracy du modèle déployé sur ses données d'entraînement : {accuracy_score(df['sejour_prolonge'], deployed_predictions):.3f}")
print(f"F1 du modèle déployé sur ses données d'entraînement : {f1_score(df['sejour_prolonge'], deployed_predictions):.3f}")

accuracy du modèle déployé sur ses données d'entraînement : 0.772
F1 du modèle déployé sur ses données d'entraînement : 0.683
